# Калибровка инструкций

Проверяется соблюдают ли инструкции границы, заданные определениями условий.

In [54]:
import re
import numpy as np
import pandas as pd
import pymorphy3

from src.paths import PROCESSED, DATA
from src.llm import GigaChat
from src.prompts import load_prompt

morph = pymorphy3.MorphAnalyzer()

CORPORA = {
    "yt": PROCESSED / "yt-comments.parquet",
    "tg": PROCESSED / "tg-groups.parquet",
    "rr": PROCESSED / "ru-reviews.parquet"
}
CALIBRATION = DATA / "calibration"
CALIBRATION.mkdir(parents=True, exist_ok=True)

LEVELS = ["level1", "level2", "level3"]
MODEL = "GigaChat-3-Lightning"
BATCH = 1
SEED = 42

N_PER_PHENOMENON = 8        # документов на явление в отладочной партии
N_PER_CORPUS = 60           # документов на корпус в приемочной

In [55]:
# Разговорные: общеупотребительные сокращения обычных слов

EXPANSIONS = {
    "щас": "сейчас", "ща": "сейчас", "че": "что", "чё": "что", "шо": "что",
    "чето": "что-то", "чёт": "что-то", "оч": "очень", "норм": "нормально",
    "ваще": "вообще", "вообщем": "в общем", "крч": "короче", "тока": "только",
    "типо": "типа", "тоесть": "то есть", "тк": "так как", "пж": "пожалуйста",
    "плз": "пожалуйста", "спс": "спасибо", "кмк": "как мне кажется"
}

WORD = re.compile(r"[А-Яа-яЁё]+")
CAPS = re.compile(r"(?<![А-Яа-яЁё])[А-ЯЁ]{3,}(?![А-Яа-яЁё])")

PHENOMENA = {
    "капс": CAPS.pattern,
    "удлиненные": r"([А-Яа-яЁё])\1{2,}",
    "ряд знаков": r"[!?]{2,}|\.{3,}",
    "эмодзи": "[\U0001F300-\U0001FAFF\u2600-\u27BF]",
    "смайлы": r"[:;=][-^]?[)(DdPp]|\)\)+|\(\(+",
    "числа": r"\d",
    "отрицания": r"\b(не|ни|нет|нельзя|никогда|никто|ничего)\b",
    "сокращения": r"\b(" + "|".join(EXPANSIONS) + r")\b",
    "обращение на ты": r"\b(ты|тебя|тебе|твой|твоя|тво[ие])\b",
    "вопрос": r"\?",
}

has = lambda text, pattern: bool(re.search(pattern, str(text)))

In [56]:
def levenshtein(a, b):
    if abs(len(a) - len(b)) > 4:
        return 9
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def tokens(text):
    return [w.lower().replace("ё", "е") for w in WORD.findall(str(text))]


def lemmas(text):
    return {morph.parse(w)[0].normal_form.replace("ё", "е") for w in tokens(text)}


def replacements(before, after):
    gone = lemmas(before) - lemmas(after)
    added = lemmas(after) - lemmas(before)

    allowed = set()
    low = str(before).lower()
    for short, full in EXPANSIONS.items():
        if re.search(rf"\b{short}\b", low):
            allowed |= lemmas(full)

    result = []
    for word in added - allowed:
        if len(word) <= 3:
            continue
        limit = 2 if len(word) <= 5 else 4
        if not any(levenshtein(word, old) <= limit for old in gone):
            result.append(word)
    return result


def caps_split(text):
    abbreviations, emphatic = [], []
    for word in CAPS.findall(str(text)):
        low = word.lower()
        tag = morph.parse(low)[0].tag
        if (not morph.word_is_known(low) or "Abbr" in tag
                or "Name" in tag or "Geox" in tag):
            abbreviations.append(low)
        else:
            emphatic.append(low)
    return abbreviations, emphatic


def short_tokens(text):
    return {w for w in tokens(text)
            if 2 <= len(w) <= 5 and w not in EXPANSIONS
            and not morph.word_is_known(w)}


def numbers(text):
    return {int(x) for x in re.findall(r"\d+", str(text)) if int(x) >= 10}


def negated(text):
    return bool(re.search(r"\b(не|ни|нет|нельзя|никогда|никто|ничего|без)\b|\b(не|без)[а-яё]{3,}", str(text).lower()))


def content_words(text):
    return [w for w in tokens(text) if len(w) > 3]


def show(title, items, limit=10):
    print(f"\n{title}: {len(items)}")
    for uid, before, after, note in items[:limit]:
        print(f"    [{uid}] {note}")
        print(f"    было:  {str(before)[:130]}")
        print(f"    стало: {str(after)[:130]}")

In [57]:
# Загрузка корпусов и разметка явлений

frames = []
for name, path in CORPORA.items():
    d = pd.read_parquet(path)
    d["corpus"] = name
    frames.append(d)

pool = pd.concat(frames, ignore_index=True)
pool["uid"] = pool.corpus + "-" + pool.id.astype(str)
pool["n_words"] = pool.text.map(lambda t: len(tokens(t)))

for label, pattern in PHENOMENA.items():
    pool[label] = pool.text.map(lambda t, p=pattern: has(t, p))
pool["аббревиатура"] = pool.text.map(lambda t: len(caps_split(t)[0]) > 0)
pool["предметное сокращение"] = pool.text.map(lambda t: len(short_tokens(t)) > 0)

print(f"документов в трех корпусах: {len(pool)}\n")
print(pool.corpus.value_counts().to_string())
print("\nдоля документов с явлением:")
targets = list(PHENOMENA) + ["аббревиатура", "предметное сокращение"]
for label in targets:
    print(f"    {label:30s} {pool[label].mean():10.1%}")

документов в трех корпусах: 406251

corpus
tg    298738
rr     69976
yt     37537

доля документов с явлением:
    капс                                 1.4%
    удлиненные                           0.6%
    ряд знаков                           4.7%
    эмодзи                               2.8%
    смайлы                               4.0%
    числа                               15.0%
    отрицания                           30.8%
    сокращения                           2.6%
    обращение на ты                      5.9%
    вопрос                              13.7%
    аббревиатура                         0.8%
    предметное сокращение               16.1%


## Партии

**Отладочная** партия отбирается по покрытию явлений и по краям диапазона длины.

**Приемочная партия** случайна и равномерна по корпусам.

In [58]:
rng = np.random.default_rng(SEED)
picked = []

for label in targets:
    available = pool[pool[label] & ~pool.uid.isin(picked)]
    if available.empty:
        print(f"нет документов с явлением: {label}")
        continue
    take = available.sample(min(N_PER_PHENOMENON, len(available)), random_state=SEED)
    picked.extend(take.uid.tolist())

rest = pool[~pool.uid.isin(picked)]
picked.extend(rest.nsmallest(3, "n_words").uid.tolist())
picked.extend(rest.nlargest(3, "n_words").uid.tolist())

debug = pool[pool.uid.isin(picked)].reset_index(drop=True)
print(f"\nотладочная партия: {len(debug)} документов")
print(debug.corpus.value_counts().to_string())
print("\nпокрытие явлений:")
for label in targets:
    print(f"    {label:30s} {int(debug[label].sum()):5d}")


отладочная партия: 102 документов
corpus
tg    65
rr    20
yt    17

покрытие явлений:
    капс                              19
    удлиненные                         8
    ряд знаков                        22
    эмодзи                            10
    смайлы                            13
    числа                             28
    отрицания                         34
    сокращения                         8
    обращение на ты                   16
    вопрос                            22
    аббревиатура                      12
    предметное сокращение             32


In [59]:
parts = []
for name in CORPORA:
    subset = pool[(pool.corpus == name) & ~pool.uid.isin(picked)]
    parts.append(subset.sample(min(N_PER_CORPUS, len(subset)), random_state=SEED))
acceptance = pd.concat(parts, ignore_index=True)

for label in targets:
    if acceptance[label].sum() >= 5:
        continue
    extra = pool[pool[label] & ~pool.uid.isin(acceptance.uid) & ~pool.uid.isin(picked)]
    if not extra.empty:
        acceptance = pd.concat(
            [acceptance, extra.sample(min(5, len(extra)), random_state=SEED)],
            ignore_index=True)

acceptance = acceptance.reset_index(drop=True)
print(f"приемочная партия: {len(acceptance)} документов")
print(acceptance.corpus.value_counts().to_string())

columns = ["uid", "corpus", "id", "text", "n_words"]
debug[columns].to_parquet(CALIBRATION / "debug.parquet", index=False)
acceptance[columns].to_parquet(CALIBRATION / "acceptance.parquet", index=False)

приемочная партия: 200 документов
corpus
tg    70
yt    65
rr    65


## Запуск

In [60]:
llm = GigaChat(model=MODEL)
PARTY = debug                  # для приемочной - acceptance

work = PARTY[["text"]].copy()
work.insert(0, "id", range(1, len(work) + 1))
to_uid = dict(zip(work.id, PARTY.uid))

results = {"orig": PARTY.set_index("uid").text}
usage = []
for level in LEVELS:
    res, log = llm.ask_batch(work, load_prompt(level), batch=BATCH, temperature=0.0)
    results[level] = res.assign(uid=res.id.map(to_uid)).set_index("uid").llm_text
    usage.append({"условие": level, "получено": int(res.llm_text.notna().sum()),
                  "отправлено": len(res), "токенов": int(log.tokens.sum()),
                  "секунд": round(float(log.elapsed.sum()))})

out = pd.DataFrame(results)

pairs = out.dropna()
print(pd.DataFrame(usage).to_string(index=False))
print(f"\nобработаны на всех условиях: {len(pairs)} из {len(out)}")

  0%|          | 0/102 [00:00<?, ?it/s]

  0%|          | 0/102 [00:00<?, ?it/s]

  0%|          | 0/102 [00:00<?, ?it/s]

условие  получено  отправлено  токенов  секунд
 level1        96         102     8010      34
 level2        95         102     8345      33
 level3        94         102     8152      34

обработаны на всех условиях: 93 из 102


## Проверка 1. Комплектность

Документы без ответа.

In [61]:
print(f"обработаны на всех условиях: {len(pairs)} из {len(out)} ({len(pairs) / len(out):.0%})")

blocked = sorted(set(out.index) - set(pairs.index))
if blocked:
    by_corpus = pd.Series([u.split("-")[0] for u in blocked]).value_counts().to_dict()
    print(f"\nне обработано: {len(blocked)} ({len(blocked) / len(out):.0%}), по корпусам {by_corpus}")
    print("по условиям:")
    for level in LEVELS:
        missing = out[out[level].isna()].index.tolist()
        print(f"    {level}: {len(missing)}")
    print("\nтексты:")
    for uid in blocked[:15]:
        levels = [l for l in LEVELS if pd.isna(out.loc[uid, l])]
        print(f"    [{uid}] нет на {', '.join(levels)}")
        print(f"       {str(out.loc[uid, 'orig'])[:110]}")

обработаны на всех условиях: 93 из 102 (91%)

не обработано: 9 (9%), по корпусам {'tg': 6, 'yt': 3}
по условиям:
    level1: 6
    level2: 7
    level3: 8

тексты:
    [tg-15698] нет на level2
       Обычно впн не там хостят
    [tg-242541] нет на level1, level2, level3
       Кек ты че выебываешься хохол мля
    [tg-256481] нет на level1, level2, level3
       И застаивили фальсифицировать выборы в США
    [tg-277597] нет на level3
       Я завтра с женой и дитё тоже на линейку, а потом если успеем в ЗАГС на развод подавать...
    [tg-4449] нет на level3
       @user_14
Напомни пожалуйста, ты как-то писал статью на хабре про развертывание приложений на AWS. Можешь дать 
    [tg-86155] нет на level1, level2, level3
       Это предприятие, которым владеет ЦРУ, добавляющая седативные препараты в мороженое, что-бы сдерживать людей от
    [yt-22844] нет на level1, level2, level3
       Первым чем вас уничтожают H2O первой уничтожат африку ,а затем и других . Колонистам нужны ресурсы Все гл

## Проверка 2. Профиль правок

Типы правки учитываются раздельно.

In [62]:
rows = []
for level in LEVELS:
    spelling = sum(1 for a, b in zip(pairs.orig, pairs[level])
                   if tokens(a) != tokens(b) and not replacements(a, b))
    lexical = sum(1 for a, b in zip(pairs.orig, pairs[level]) if replacements(a, b))
    grew = sum(1 for a, b in zip(pairs.orig, pairs[level])
               if len(content_words(b)) > len(content_words(a)) * 1.05)
    shrank = sum(1 for a, b in zip(pairs.orig, pairs[level])
                 if len(content_words(b)) < len(content_words(a)) * 0.9)
    ratio = np.median([len(tokens(b)) / max(len(tokens(a)), 1)
                       for a, b in zip(pairs.orig, pairs[level])])
    rows.append({"условие": level, "правка написания": spelling,
                 "замена слов": lexical, "прирост слов": grew,
                 "утрата слов": shrank, "отношение длин": round(float(ratio), 2)})
print(pd.DataFrame(rows).to_string(index=False))

условие  правка написания  замена слов  прирост слов  утрата слов  отношение длин
 level1                18            2             0            3            1.00
 level2                27            5             0            3            1.00
 level3                 4           87            58            8            1.08


## Проверка 3. Условия 1 и 2 не изменяют словарь

In [63]:
for level in ["level1", "level2"]:
    found = [(uid, pairs.loc[uid, "orig"], pairs.loc[uid, level],
              ", ".join(replacements(pairs.loc[uid, "orig"], pairs.loc[uid, level])))
             for uid in pairs.index
             if replacements(pairs.loc[uid, "orig"], pairs.loc[uid, level])]
    show(f"{level}: замена слов", found)


level1: замена слов: 2
    [tg-28085] быть
    было:  А, у тебя? Тогда хз
    стало: А, у тебя? Тогда х**е**
    [tg-228683] пизда
    было:  не пиздить же будут тебя
    стало: не пизд*ть же будут тебя

level2: замена слов: 5
    [yt-12] биологический
    было:  Хакеры могут создать намного больше вреда, чем ИИ, а потом скрыться. У них и стимул для этого есть потому, что они тоже био органи
    стало: Хакеры могут создать намного больше вреда, чем ИИ, а потом скрыться. У них и стимул для этого есть потому, что они тоже биологичес
    [yt-1859] така
    было:  вы сейчас говорите словами такачих в 1900 году когда внедряли прядильные станки. или когда заменили извозчиков на машины ДВС.
    стало: вы сейчас говорите словами така'чих в 1900 году, когда внедряли прядильные станки. или когда заменили извозчиков на машины ДВС.
    [tg-154998] фастфуд
    было:  ещё возник вопросик, как правильно сделать - есть фастапи с бд-шкой, нужно одно поле сделать enum...это нормальный подход..или луч
 

## Проверка 4. Условия 1 и 2 не изменяют оформление

In [64]:
rows = []
for column in ["orig"] + LEVELS:
    texts = pairs[column]
    rows.append({
        "условие": column,
        "документов": len(texts),
        "аббревиатуры": sum(len(caps_split(t)[0]) for t in texts),
        "эмфатический капс": sum(len(caps_split(t)[1]) for t in texts),
        "эмодзи": int(texts.map(lambda t: has(t, PHENOMENA["эмодзи"])).sum()),
        "смайлы": int(texts.map(lambda t: has(t, PHENOMENA["смайлы"])).sum()),
        "ряды знаков": int(texts.map(lambda t: has(t, PHENOMENA["ряд знаков"])).sum())
    })
print(pd.DataFrame(rows).to_string(index=False))

lost_caps = [(uid, pairs.loc[uid, "orig"], pairs.loc[uid, "level2"], "снят капс")
             for uid in pairs.index
             if len(caps_split(pairs.loc[uid, "orig"])[1])
             > len(caps_split(pairs.loc[uid, "level2"])[1])]
show("level2: эмфатический капс снят", lost_caps)

условие  документов  аббревиатуры  эмфатический капс  эмодзи  смайлы  ряды знаков
   orig          93             9                 27       8      13           19
 level1          93             7                 26       8      13           19
 level2          93             7                  5       8      12           18
 level3          93             3                  0       0       1            1

level2: эмфатический капс снят: 5
    [yt-42] снят капс
    было:  ИИ ЭТО МОШЕННИЧЕСТВО
    стало: ИИ — это мошенничество.
    [yt-6606] снят капс
    было:  Я ВАС ПОНЕМАЮ
    стало: Я вас пойму.
    [rr-820] снят капс
    было:  ПЛАТЬЕ ХОРОШЕЕ НО ЭТО НЕ ХЛОПОК . ТКАНЬ АКРИЛ ИЛИ ЧТО ТО ВРОДЕ ЭТОГО.
    стало: ПЛАТЬЕ ХОРОШЕЕ, но это не хлопок. Ткань — акрил или что-то вроде этого.
    [rr-32326] снят капс
    было:  РАЗМЕР НЕ ПОДОШЁЛ ХОТЯ Я ЭТОТ РАЗМЕР И НОШУ МАТЕРИАЛ СИЛЬНО ТОНКИЙ
    стало: Размер не подошёл, хотя я этот раз размер и ношу. Материал сильно тонкий.
    [rr-46785] сня

## Проверка 5. Условие 2 не достраивает содержание

In [65]:
added = [(uid, pairs.loc[uid, "orig"], pairs.loc[uid, "level2"],
          f"{len(content_words(pairs.loc[uid, 'orig']))} -> "
          f"{len(content_words(pairs.loc[uid, 'level2']))} слов")
         for uid in pairs.index
         if len(content_words(pairs.loc[uid, "level2"]))
         > len(content_words(pairs.loc[uid, "orig"])) * 1.05]
show("level2: прирост содержания", added)


level2: прирост содержания: 0


## Проверка 6. Условие 3 выполняет формализацию

In [66]:
rows = []
for column in ["orig"] + LEVELS:
    texts = pairs[column]
    rows.append({
        "условие": column,
        "удлиненные": int(texts.map(lambda t: has(t, PHENOMENA["удлиненные"])).sum()),
        "эмфатический капс": sum(len(caps_split(t)[1]) for t in texts),
        "эмодзи": int(texts.map(lambda t: has(t, PHENOMENA["эмодзи"])).sum()),
        "замена слов": sum(1 for a, b in zip(pairs.orig, texts) if replacements(a, b))
    })
print(pd.DataFrame(rows).to_string(index=False))

условие  удлиненные  эмфатический капс  эмодзи  замена слов
   orig           7                 27       8            0
 level1           6                 26       8            2
 level2           5                  5       8            5
 level3           0                  0       0           87


## Проверка 7. Сохранность содержания

In [67]:
rows = []
for level in LEVELS:
    rows.append({
        "условие": level,
        "утрачены числа": sum(1 for a, b in zip(pairs.orig, pairs[level])
                              if numbers(a) - numbers(b)),
        "утрачено отрицание": sum(1 for a, b in zip(pairs.orig, pairs[level])
                                  if negated(a) and not negated(b)),
        "пересказ со стороны": int(pairs[level].str.contains(
            r"\b(?:автор|говорящий|пишущий)\s+(?:пишет|отмечает|указывает|считает)",
            regex=True).sum())
    })
print(pd.DataFrame(rows).to_string(index=False))

for level in LEVELS:
    lost = [(uid, pairs.loc[uid, "orig"], pairs.loc[uid, level],
             f"пропало {sorted(numbers(pairs.loc[uid, 'orig']) - numbers(pairs.loc[uid, level]))}")
            for uid in pairs.index
            if numbers(pairs.loc[uid, "orig"]) - numbers(pairs.loc[uid, level])]
    show(f"{level}: утрачены числа", lost)

for level in LEVELS:
    lost = [(uid, pairs.loc[uid, "orig"], pairs.loc[uid, level], "утрачено отрицание")
            for uid in pairs.index
            if negated(pairs.loc[uid, "orig"]) and not negated(pairs.loc[uid, level])]
    show(f"{level}: утрачено отрицание", lost)

условие  утрачены числа  утрачено отрицание  пересказ со стороны
 level1               1                   0                    0
 level2               0                   0                    0
 level3               3                   4                    0

level1: утрачены числа: 1
    [tg-34092] пропало [20]
    было:  всем привет, можете помочь решить одну задачу если не сложно, уже 20 минут пытаюсь ее решить
    стало: всем привет, можете помочь решить одну задачу если не сложно, уже оч пытаюсь ее решить

level2: утрачены числа: 0

level3: утрачены числа: 3
    [yt-1859] пропало [1900]
    было:  вы сейчас говорите словами такачих в 1900 году когда внедряли прядильные станки. или когда заменили извозчиков на машины ДВС.
    стало: Вы сейчас используете выражения, которые были характерны для начала XX века, когда внедрялись прядильные станки или когда извозчик
    [tg-274819] пропало [10]
    было:  (2 часа ебался с роутингами, хз что делал, заебался, закрыл ИДЕ, открыл сайт реак

## Проверка 8.

In [68]:
addressed = lambda t: has(t, PHENOMENA["обращение на ты"])
question = lambda t: has(t, PHENOMENA["вопрос"])

rows = []
for level in LEVELS:
    rows.append({
        "условие": level,
        "смена обращения": sum(1 for a, b in zip(pairs.orig, pairs[level])
                               if addressed(a) and not addressed(b)),
        "вопрос стал утверждением": sum(1 for a, b in zip(pairs.orig, pairs[level])
                                        if question(a) and not question(b)),
        "приведён регистр": sum(1 for a, b in zip(pairs.orig, pairs[level])
                                if len(caps_split(a)[1]) > len(caps_split(b)[1])),
        "заглавных осталось": sum(len(caps_split(t)[1]) for t in pairs[level]),
    })
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nзаглавных в исходных текстах: "
      f"{sum(len(caps_split(t)[1]) for t in pairs.orig)}")

условие  смена обращения  вопрос стал утверждением  приведён регистр  заглавных осталось
 level1                0                         0                 1                  26
 level2                0                         0                 5                   5
 level3               11                         5                 8                   0

заглавных в исходных текстах: 27


## Тексты

Выборочный просмотр.

In [69]:
SHOW = 15
for uid in pairs.index[:SHOW]:
    print("=" * 80)
    print(f"[{uid}]")
    for column in ["orig"] + LEVELS:
        print(f"    {column:7s} {str(out.loc[uid, column])[:150]}")
    print()

[yt-12]
    orig    Хакеры могут создать намного больше вреда, чем ИИ, а потом скрыться. У них и стимул для этого есть потому, что они тоже био организмы. И спрятаться в 
    level1  Хакеры могут создать намного больше вреда, чем ИИ, а потом скрыться. У них и стимул для этого есть потому, что они тоже био организмы. И спрятаться в 
    level2  Хакеры могут создать намного больше вреда, чем ИИ, а потом скрыться. У них и стимул для этого есть потому, что они тоже биологические организмы. И спр
    level3  Хакеры способны нанести больший ущерб, чем искусственный интеллект. У них есть для этого мотивация, поскольку они являются биологическими существами и

[yt-27]
    orig    ... а кибернетика - лженаука )))
    level1  ... а кибернетика - лженаука )))
    level2  ... а кибернетика - лженаука )))
    level3  Кибернетика рассматривается как лженаука.

[yt-42]
    orig    ИИ ЭТО МОШЕННИЧЕСТВО
    level1  ИИ ЭТО МОШЕННИЧЕСТВО
    level2  ИИ — это мошенничество.
    level3  Искусственный интел

## Пороги

In [70]:
LIMITS = {
    "замена слов": 0.10,       
    "прирост содержания": 0.10,
    "утрата чисел": 0.10,      
    "сохранение оформления": 0.80,
    "снятие оформления": 0.10,
    "формализация": 0.70,
    "комплектность": 0.85,
    "отношение длин": 1.5
}


def check(name, value, passed):
    print(f"{'прошел' if passed else 'НЕ прошел':>10}  {name}: {value}")


n = len(pairs)
texts = {c: pairs[c] for c in ["orig"] + LEVELS}
count = lambda level, label: int(texts[level].map(
    lambda t: has(t, PHENOMENA[label])).sum())
caps_count = lambda level, index: sum(len(caps_split(t)[index]) for t in texts[level])

returned = len(pairs) / len(out)
check("комплектность", f"{returned:.0%}", returned >= LIMITS["комплектность"])

for level in ["level1", "level2"]:
    lexical = sum(1 for a, b in zip(pairs.orig, pairs[level]) if replacements(a, b))
    check(f"{level}: словарь не изменен", f"{lexical} из {n} ({lexical / n:.0%})",
          lexical / n <= LIMITS["замена слов"])

grew = sum(1 for a, b in zip(pairs.orig, pairs.level2)
           if len(content_words(b)) > len(content_words(a)) * 1.05)
check("level2: содержание не достроено", f"{grew} из {n} ({grew / n:.0%})",
      grew / n <= LIMITS["прирост содержания"])

for index, what in [(0, "аббревиатуры"), (1, "эмфатический капс")]:
    before = caps_count("orig", index)
    if not before:
        continue
    after = caps_count("level1", index)
    check(f"level1: {what} сохранены", f"{before} -> {after}",
          after >= LIMITS["сохранение оформления"] * before)

if caps_count("orig", 1):
    check("level3: эмфатический капс снят",
          f"{caps_count('orig', 1)} -> {caps_count('level3', 1)}",
          caps_count("level3", 1) <= LIMITS["снятие оформления"] * caps_count("orig", 1))

for label, level, mode in [("эмодзи", "level1", "сохранены"),
                           ("эмодзи", "level2", "сохранены"),
                           ("эмодзи", "level3", "сняты")]:
    before = count("orig", label)
    if before == 0:
        print(f"{'пропущен':>10}  {label} на {level}: явления нет в партии")
        continue
    after = count(level, label)
    passed = (after >= LIMITS["сохранение оформления"] * before if mode == "сохранены"
              else after <= LIMITS["снятие оформления"] * before)
    check(f"{level}: {label} {mode}", f"{before} -> {after}", passed)

formalized = sum(1 for a, b in zip(pairs.orig, pairs.level3) if replacements(a, b))
check("level3: формализация выполнена", f"{formalized} из {n} ({formalized / n:.0%})",
      formalized / n >= LIMITS["формализация"])

lost = sum(1 for a, b in zip(pairs.orig, pairs.level3) if numbers(a) - numbers(b))
check("level3: числа сохранены", f"утрачено {lost} из {n} ({lost / n:.0%})",
      lost / n <= LIMITS["утрата чисел"])

retelling = int(pairs.level3.str.contains(
    r"\b(?:автор|говорящий|пишущий)\s+(?:пишет|отмечает|указывает|считает)",
    regex=True).sum())
check("level3: пересказа со стороны нет", retelling, retelling == 0)

ratio = np.median([len(tokens(b)) / max(len(tokens(a)), 1)
                   for a, b in zip(pairs.orig, pairs.level3)])
check("level3: отношение длин", round(float(ratio), 2), ratio <= LIMITS["отношение длин"])

    прошел  комплектность: 91%


    прошел  level1: словарь не изменен: 2 из 93 (2%)
    прошел  level2: словарь не изменен: 5 из 93 (5%)
    прошел  level2: содержание не достроено: 0 из 93 (0%)
 НЕ прошел  level1: аббревиатуры сохранены: 9 -> 7
    прошел  level1: эмфатический капс сохранены: 27 -> 26
    прошел  level3: эмфатический капс снят: 27 -> 0
    прошел  level1: эмодзи сохранены: 8 -> 8
    прошел  level2: эмодзи сохранены: 8 -> 8
    прошел  level3: эмодзи сняты: 8 -> 0
    прошел  level3: формализация выполнена: 87 из 93 (94%)
    прошел  level3: числа сохранены: утрачено 3 из 93 (3%)
    прошел  level3: пересказа со стороны нет: 0
    прошел  level3: отношение длин: 1.08


In [ ]:
# Сохранение ответов

RUN = 1

name = f"acceptance_{MODEL}_run{RUN}"
out.to_parquet(CALIBRATION / f"{name}.parquet")
pd.DataFrame(usage).to_csv(CALIBRATION / f"{name}_usage.csv", index=False)